In [ ]:
"""
customer support agent that can pull answers from two fundamentally different data sources in a single conversation:

1) Structured data (SQLite) - Customer records, order history, and product inventory stored in relational tables. 
The agent queries these through parameterized functions (never raw SQL).
2) Unstructured data (Knowledge Base) - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. 
The agent searches these by keyword.

    Tool	                Data Source	                                    Purpose
--------------            -------------------                   ---------------------------------------
search_orders	            SQLite orders + customers	        Look up orders by email, ID, or status
search_products	            SQLite products	                    Find products by category, keyword, or stock
search_knowledge_base	    Markdown docs	                    Search policies, FAQs, and documentation

The key insight is that the model never generates SQL directly. 
Each tool accepts structured parameters (email, order ID, category, etc.) and the Python function constructs the appropriate 
query internally. This is safer and more predictable than text-to-SQL approaches.
"""

In [12]:
import os
from getpass import getpass
import json
import sqlite3
import pathlib
from typing import Literal
import sys

from llm_config import ollama, MODEL_OLLAMA
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

In [13]:
# Sample data (JSON): customers, products, and orders
# Knowledge base (Markdown files): return policy, shipping info, FAQ, pricing
# System instructions (text file): the prompt that tells the agent how to behave

In [18]:
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
RESOURCES = pathlib.Path("..","resources","knowledge_base")
data = json.loads((RESOURCES / "data" / "customer_support_sample.json").read_text())
print(f"Loaded {len(data['customers'])} customers, "
      f"{len(data['products'])} products, "
      f"{len(data['orders'])} orders")

 
knowledge_base = {}
for md_file in sorted((RESOURCES).glob("*.md")):     # Find every file whose name ends with .md.
    knowledge_base[md_file.name] = md_file.read_text()                           # uses the filename as the dictionary key and the file contents as the value.
print(f"\nKnowledge base contains {len(knowledge_base)} documents:")
for filename, content in knowledge_base.items():
    line_count = len(content.strip().splitlines())
    print(f"  - {filename} ({line_count} lines)")

"""
knowledge_base = {
    "payments.md": "Payment information....................data from file as a value of the key",
    "returns.md": "Returns Policy\n\nCustomers can return................data from file as a value of the key",
    "shipping.md": "Shipping information...data from file as a value of the key"
    
    Markdown files
        ↓
    Read files
        ↓
    Python dictionary
        ↓
    AI Agent / RAG
}
"""

instructions = (RESOURCES / "prompts" / "customer_support_agent_instructions.txt").read_text()
print(f"\nSystem instructions ({len(instructions)} chars):")
print(instructions[:200])


Loaded 8 customers, 10 products, 10 orders

Knowledge base contains 4 documents:
  - faq.md (16 lines)
  - pricing_plans.md (22 lines)
  - return_policy.md (19 lines)
  - shipping_info.md (17 lines)

System instructions (314 chars):
You are a helpful customer support agent for an online store.
Use the available tools to look up information before answering.
Always cite which source you used (order database, product catalog, or kn
